In [1]:
!pip install causal-conv1d mamba-ssm --no-build-isolation

  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.4/216.4 kB 4.9 MB/s eta 0:00:0000:01
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 46.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/728.5 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 MB 21.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.4/772.4 kB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 67.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 103.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 14.3 MB/s e

In [2]:
!git clone https://github.com/Ghanasree-S/BVI-Net-ISIC.git
%cd BVI-Net-ISIC
!pip install -q -r requirements.txt

Cloning into 'BVI-Net-ISIC'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (39/39), done.
remote: Total 55 (delta 18), reused 51 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 64.64 KiB | 3.40 MiB/s, done.
Resolving deltas: 100% (18/18), done.
/kaggle/working/BVI-Net-ISIC


In [3]:
!bash data/download_isic_official_val.sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  228M  100  228M    0     0  73.4M      0  0:00:03  0:00:03 --:--:-- 73.5M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  741k  100  741k    0     0  2159k      0 --:--:-- --:--:-- --:--:-- 2162k
Extracting...
Done. Official validation data in: data/isic2018_val_raw


In [4]:
!python data/prepare_isic_a.py \
  --raw_dir /kaggle/input/datasets/tschandl/isic2018-challenge-task1-data-segmentation \
  --val_raw_dir data/isic2018_val_raw \
  --out_dir data/isic2018a \
  --fraction 1.0

ISIC2018 training pool total: 2594 | using: 2594
Official validation set found: 100 images (matches paper's split)
Train: 2335 (official pool minus self-carved test) | Val: 100 (official) | Test: 259 (self-carved -- paper's real test masks are not public)
resizing + splitting: 100%|█████████████████| 2694/2694 [07:27<00:00,  6.02it/s]
Done. Dataset saved to: data/isic2018a


In [7]:
!git pull origin main
!python width_search.py --data_dir data/isic2018a --epochs 20 --batch_size 8

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 563 bytes | 563.00 KiB/s, done.
From https://github.com/Ghanasree-S/BVI-Net-ISIC
 * branch            main       -> FETCH_HEAD
   4e6b42f..7fafd85  main       -> origin/main
Updating 4e6b42f..7fafd85
Fast-forward
 width_search.py | 3 ++-
 1 file changed, 2 insertions(+), 1 deletion(-)
GPU memory free: 15.53 GB / 15.64 GB total
Using device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)

=== current (8-16-32-64-128, N=32) ===
Parameters: 777,584 (0.7776M)
  epoch 01 | train_dice=0.6208 val_dice=0.7899
  epoch 02 | train_dice=0.7424 val_dice=0.7602
  epoch 03 | train_dice=0.7664 

In [13]:
!git pull origin main
!python train.py --data_dir data/isic2018a --epochs 50 --batch_size 8 --lr 0.001 --patience 15 \
  --channels 4 8 8 16 16 --gcn_nodes 8

From https://github.com/Ghanasree-S/BVI-Net-ISIC
 * branch            main       -> FETCH_HEAD
Already up to date.
Using device: cuda
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Model parameters: 27,312 (0.0273M)
Epoch 01 | train_loss=1.0680 train_dice=0.7045 | val_loss=0.7282 val_dice=0.8277 | lr=0.001000
  -> new best (val_dice=0.8277), checkpoint saved
Epoch 02 | train_loss=0.7118 train_dice=0.7846 | val_loss=0.5002 val_dice=0.8356 | lr=0.001000
  -> new best (val_dice=0.8356), checkpoint saved
Epoch 03 | train_loss=0.5634 train_dice=0.7997 | val_loss=0.4504 val_dice=0.8398 | lr=0.001000
  -> new best (val_dice=0.8398), checkpoint saved
Epoch 04 | train_loss=0.4962 train_dice=0.8139 | val_loss=0.4310 val_dice=0.8378 | lr=0.001000
Epoch 05 | train_loss=0.4615 train_dice=0.8232 | val_loss=0.4054 val_dice=

In [14]:
!python evaluate.py --checkpoint checkpoints/best.pt --data_dir data/isic2018a --visualize \
  --channels 4 8 8 16 16 --gcn_nodes 8

evaluating: 100%|█████████████████████████████| 259/259 [00:11<00:00, 22.29it/s]

=== Test set results (ISIC2018-a) ===
        dice: 0.8631
        miou: 0.7855
    accuracy: 0.9544
 specificity: 0.9780
 sensitivity: 0.8817
        assd: 6.8761
